# Data Preprocessing
De volgende code wordt gebruikt om de datasets te preprocessen tot bestanden die worden gebruikt voor het data story-project.

In [1]:
# Step 1: Import necessary libraries
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy.stats import pearsonr

# CSV inlezen


# Step 2: Load and filter the datasets
df_1 = pd.read_csv("/Users/diazwiersma/data-story/_static/Graphs/remote_work_productivity.csv")
df_2 = pd.read_csv("/Users/diazwiersma/data-story/_static/Graphs/The Impacts of Working Remotely and in an Office Survey.csv")
df_3 = pd.read_csv('/Users/diazwiersma/data-story/_static/Graphs/Impact_of_Remote_Work_on_Mental_Health.csv')
df_4 = pd.read_csv('/Users/diazwiersma/data-story/_static/Graphs/Extended_Employee_Performance_and_Productivity_Data.csv')

# Step 3: Merge datasets

# Step 4: Process the data for visualizations

# Figuur 1
avg_productivity = df_1.groupby('Employment_Type')['Productivity_Score'].mean()
## Graph
avg_productivity.plot(kind='bar', color=['skyblue', 'salmon'])
plt.title("Gemiddelde Productiviteit: Remote vs Office")
plt.ylabel("Productivity Score")
plt.xlabel("Employment Type")
plt.ylim(0, 100)
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig("/Users/diazwiersma/data-story/_static/Graphs/figure_1.png")  # Opslaan voor hergebruik
plt.close()


# Figuur 2
sizes = [(df_2["Do you think that working from home increases your work productivity?"] == "Yes").mean() * 100,]
sizes.append(100 - sizes[0])  
labels = ["Ja", "Nee"]
colors = ['salmon', 'skyblue']
plt.figure(figsize=(8,7))
wedges, texts, autotexts = plt.pie(sizes, colors=colors, autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
plt.title("Denk je dat thuiswerken je productiviteit op werk verhoogt?")
plt.axis('equal') 
plt.legend(wedges, labels, title="Antwoorden", loc="center left", bbox_to_anchor=(1, 0, 0.5, 1))
plt.tight_layout()
plt.savefig("/Users/diazwiersma/data-story/_static/Graphs/figure_2.png") 
plt.close()

#Figuur 3
# Scatterplot
sns.scatterplot(
    data=df_1,
    x='Hours_Worked_Per_Week',
    y='Productivity_Score',
    hue='Employment_Type'
)
plt.title('Relatie tussen gewerkte uren en productiviteit per werkvorm')
plt.xlabel('Aantal gewerkte uren per week')
plt.ylabel('Productiviteitsscore')
plt.legend()
plt.tight_layout()
plt.savefig("/Users/diazwiersma/data-story/_static/Graphs/figure_3.png") 
plt.close()

#Figuur 4
## Clean
df_cleaned = df_3.copy()
df_cleaned.columns = df_cleaned.columns.str.strip()
df_cleaned['Work_Location'] = df_cleaned['Work_Location'].str.strip().str.capitalize()
df_cleaned['Stress_Level'] = df_cleaned['Stress_Level'].str.strip().str.capitalize()
df_cleaned['Mental_Health_Condition'] = df_cleaned['Mental_Health_Condition'].fillna('None').str.strip().str.capitalize()
df_cleaned['Sleep_Quality'] = df_cleaned['Sleep_Quality'].str.strip().str.capitalize()
## Processing
locations = df_cleaned['Work_Location'].unique()
indicators = ['Lage stress', 'Geen mentale aandoening', 'Goede slaapkwaliteit']
data = []
for location in locations:
    subset = df_cleaned[df_cleaned['Work_Location'] == location]
    total = len(subset)
    low_stress = (subset['Stress_Level'] == 'Low').sum() / total * 100
    no_mental_issue = (subset['Mental_Health_Condition'] == 'None').sum() / total * 100
    good_sleep = (subset['Sleep_Quality'] == 'Good').sum() / total * 100
    data.append([low_stress, no_mental_issue, good_sleep])
data = np.array(data).T  
y = np.arange(len(locations))
bar_height = 0.2
colors = ['mediumseagreen', 'cornflowerblue', 'gold']
## Plotting
plt.figure(figsize=(10, 6))
for i in range(len(indicators)):
    bars = plt.barh(y + i * bar_height, data[i], height=bar_height, label=indicators[i], color=colors[i])
    for bar in bars:
        width = bar.get_width()
        plt.text(width + 1, bar.get_y() + bar.get_height()/2,
                 f'{width:.1f}%', va='center', fontsize=9)
plt.xlabel("Percentage respondenten")
plt.ylabel("Werklocatie")
plt.xlim(0, 100)
plt.yticks(y + bar_height, locations)
plt.title("Welzijnsindicatoren per werklocatie")
plt.legend()
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig("/Users/diazwiersma/data-story/_static/Graphs/figure_4.png") 
plt.close()

# Figure 5
avg_productivity = df_1.groupby('Employment_Type')['Well_Being_Score'].mean()
## Plotting
avg_productivity.plot(kind='bar', color=['skyblue', 'salmon'])
plt.title("Gemiddeld Welzijn: Remote vs Office")
plt.ylabel("Well Being Score")
plt.xlabel("Employment Type")
plt.ylim(0, 100)
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig("/Users/diazwiersma/data-story/_static/Graphs/figure_5.png") 
plt.close()

# Figure 6
fig, axes = plt.subplots(1, 2, figsize=(12,5), sharex=True)
sns.scatterplot(data=df_1, x='Hours_Worked_Per_Week', y='Well_Being_Score', ax=axes[0])
sns.regplot(data=df_1, x='Hours_Worked_Per_Week', y='Well_Being_Score', scatter=False, color='red', ax=axes[0])
axes[0].set_title('Uren gewerkt vs Welzijn')
sns.scatterplot(data=df_1, x='Hours_Worked_Per_Week', y='Productivity_Score', ax=axes[1])
sns.regplot(data=df_1, x='Hours_Worked_Per_Week', y='Productivity_Score', scatter=False, color='red', ax=axes[1])
axes[1].set_title('Uren gewerkt vs Productiviteit')
plt.tight_layout()
plt.savefig("/Users/diazwiersma/data-story/_static/Graphs/figure_6.png")
plt.close()

# Figure 7
## Cleaning
consultants = df_4[df_4['Job_Title'] == 'Consultant'].copy()  # voorkomt SettingWithCopyWarning
bins = [30, 35, 40, 45, 50, 55, 60]
labels = ['30-34', '35-39', '40-44', '45-49', '50-54', '55-59']
## Processing
consultants['Hours_Bin'] = pd.cut(
    consultants['Work_Hours_Per_Week'],
    bins=bins,
    labels=labels,
    right=False
)
avg_salary = consultants.groupby(['Department', 'Hours_Bin'], observed=True)['Monthly_Salary'].mean().round(2).reset_index()
## Plotting
fig = px.line(
    avg_salary,
    x='Hours_Bin',
    y='Monthly_Salary',
    color='Department',
    markers=True,
    labels={
        'Hours_Bin': 'Werkuren per week',
        'Monthly_Salary': 'Gemiddeld maandsalaris (€)',
        'Department': 'Afdeling'
    },
    title='Gemiddeld maandsalaris vs werkuren per week (Consultants per afdeling)'
)
fig.update_layout(xaxis=dict(tickmode='array', tickvals=labels, ticktext=labels))
fig.write_html("/Users/diazwiersma/data-story/_static/Graphs/figure_7.html")
